# Battery Model (SOC / DoD / SOH)

In [1]:
import sys, os
sys.path.append(os.path.abspath("C:/Users/bomma/Desktop/DESKTOP_MAIN_DISP/MATHS_PROJECT_SEM5/A_project_re/project_COMPLETE_all_5_days/project/src"))
from robots import RobotFleet
from battery import BatteryModel

DATASET_ROOT =  "C:/Users/bomma/Desktop/DESKTOP_MAIN_DISP/MATHS_PROJECT_SEM5/A_project_re/project_COMPLETE_all_5_days/project/data/dataset"
fleet = RobotFleet(DATASET_ROOT)
batt = BatteryModel(fleet, DATASET_ROOT)
print(batt.summary())


UGV01  SOC= 79.3%  DoD= 20.7%  SOH=0.916
UGV02  SOC= 90.0%  DoD= 10.0%  SOH=0.893
UGV03  SOC= 65.2%  DoD= 34.8%  SOH=0.956
UGV04  SOC= 69.2%  DoD= 30.8%  SOH=0.921
USV01  SOC= 79.4%  DoD= 20.6%  SOH=0.855
USV02  SOC= 89.9%  DoD= 10.1%  SOH=0.892
USV03  SOC= 87.4%  DoD= 12.6%  SOH=0.972
USV04  SOC= 85.3%  DoD= 14.7%  SOH=0.925
UAV01  SOC= 81.4%  DoD= 18.6%  SOH=0.904
UAV02  SOC= 97.3%  DoD=  2.7%  SOH=0.995
UAV03  SOC= 78.2%  DoD= 21.8%  SOH=0.918
UAV04  SOC= 75.3%  DoD= 24.7%  SOH=0.958


## Predicted vs actual energy consumption for one task

In [2]:
r = fleet.robots["UGV01"]
distance, workload = 15.0, 6.3

predicted_energy = batt.predicted_energy_wh(r, distance, workload)
predicted_soc = batt.predicted_soc_after(r, distance, workload)
feasible = batt.is_feasible(r, distance, workload)
print(f"Before: SOC={r.soc:.1f}%  DoD={r.dod:.1f}%  SOH={r.soh:.3f}")
print(f"Predicted energy: {predicted_energy:.2f} Wh -> predicted SOC after: {predicted_soc:.1f}%  feasible={feasible}")

consumed = batt.consume(r, distance, workload)
print(f"\nAfter executing: SOC consumed={consumed:.2f}%")
print(f"After:  SOC={r.soc:.1f}%  DoD={r.dod:.1f}%  SOH={r.soh:.3f}")
assert abs(predicted_soc - r.soc) < 0.05, "predicted and actual post-task SOC should match (deterministic model)"
print("OK: predicted SOC matches actual consumption.")


Before: SOC=79.3%  DoD=20.7%  SOH=0.916
Predicted energy: 42.07 Wh -> predicted SOC after: 75.1%  feasible=True

After executing: SOC consumed=4.23%
After:  SOC=75.1%  DoD=24.9%  SOH=0.916
OK: predicted SOC matches actual consumption.


## Battery feasibility at low SOC 

In [3]:
low = fleet.robots["UGV01"]
low.soc = 18.0  # force a near-minimum SOC scenario
print(f"min_soc for {low.robot_id} = {low.min_soc:.1f}%, current SOC = {low.soc:.1f}%")
feasible_big_task = batt.is_feasible(low, distance=40.0, workload=25.0)
feasible_small_task = batt.is_feasible(low, distance=3.0, workload=1.0)
print(f"Feasible for a large task? {feasible_big_task}  (expected False: would drop below min_soc)")
print(f"Feasible for a small task? {feasible_small_task}")
assert not feasible_big_task
print("OK: low-SOC robot correctly loses feasibility for energy-heavy tasks.")


min_soc for UGV01 = 18.7%, current SOC = 18.0%
Feasible for a large task? False  (expected False: would drop below min_soc)
Feasible for a small task? False
OK: low-SOC robot correctly loses feasibility for energy-heavy tasks.


## Recharging 

In [4]:
before = low.soc
batt.recharge(low, rate=5.0, dt=3.0)
print(f"SOC after charging 3 timesteps at 5.0%/step: {before:.1f}% -> {low.soc:.1f}%")
assert low.soc > before
print("OK: recharge increases SOC. Note SOH is NOT restored by charging (permanent degradation).")


SOC after charging 3 timesteps at 5.0%/step: 18.0% -> 33.0%
OK: recharge increases SOC. Note SOH is NOT restored by charging (permanent degradation).
